# CAFA-6 Graph-Aware Protein Function Prediction

Notebook này là bản cải tiến từ `cafa6.ipynb`, tập trung tận dụng cấu trúc GO ontology:

- propagate ground-truth labels lên ancestor terms khi train/evaluate;
- thêm IA-weighted BCE để term cụ thể/hiếm có trọng số cao hơn;
- thêm hierarchical consistency penalty để hạn chế `score(child) > score(parent)`;
- post-process prediction bằng parent propagation: `parent_score = max(parent_score, child_score)`;
- xuất bảng trọng số, đánh giá flat vs graph-aware, per-aspect metrics và official evaluator inputs.

Phần model/embedding giữ cùng tinh thần notebook gốc: ESM-MLP, ProtCNN, BiLSTM-Attention và weighted ensemble.


In [ ]:
from __future__ import annotations

import json
import gc
import math
import os
import random
import re
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


In [ ]:
CONFIG = {
    # Paths: override these in Kaggle/local if needed.
    "base_dir": "/kaggle/input/cafa-6-protein-function-prediction",
    "artifact_dir": "/kaggle/working/cafa6_graph_aware_artifacts",
    "baseline_split_dir": "/kaggle/working/cafa6_high_performance_artifacts/splits",

    # Label universe.
    "max_labels": 3000,
    "min_label_freq": 8,
    "use_label_propagation": True,
    "include_ancestors_outside_selected": False,

    # Split.
    "valid_size": 0.15,
    "test_size": 0.15,
    "split_mode": "stratified_protein",

    # ESM embedding branch.
    "embedding_model_name": "facebook/esm2_t30_150M_UR50D",
    "embedding_max_length": 1022,
    "embedding_batch_size": 8,
    "embedding_pooling": "mean",
    "use_fp16_embeddings": True,
    "cache_embeddings": True,

    # Branch switches and model hyperparameters.
    "train_esm_mlp": True,
    "train_protcnn": True,
    "train_bilstm_attention": True,
    "esm_hidden_dims": [1024, 512],
    "esm_dropout": 0.35,
    "esm_epochs": 8,
    "esm_batch_size": 256,
    "sequence_max_length": 1024,
    "sequence_batch_size": 256,
    "protcnn_epochs": 5,
    "protcnn_dropout": 0.35,
    "bilstm_epochs": 8,
    "bilstm_dropout": 0.35,
    "ensemble_weights": {
        "esm_mlp": 0.5,
        "protcnn": 0.25,
        "bilstm_attention": 0.25,
    },
    "learning_rate": 2e-4,
    "weight_decay": 1e-4,
    "early_stopping_patience": 3,

    # Graph-aware loss.
    "pos_weight_clip": 30.0,
    "ia_loss_power": 0.5,
    "ia_weight_clip": 8.0,
    "hierarchy_penalty_weight": 0.15,
    "hierarchy_margin": 0.0,
    "hierarchy_edge_sample": 4096,

    # Evaluation.
    "threshold_grid": [0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50],
    "official_eval_top_k": 1500,
}

BASE_DIR = Path(CONFIG["base_dir"])
TRAIN_DIR = BASE_DIR / "Train"
ARTIFACT_DIR = Path(CONFIG["artifact_dir"])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "train_seq": TRAIN_DIR / "train_sequences.fasta",
    "train_terms": TRAIN_DIR / "train_terms.tsv",
    "train_tax": TRAIN_DIR / "train_taxonomy.tsv",
    "ia": BASE_DIR / "IA.tsv",
    "obo": TRAIN_DIR / "go-basic.obo",
}
for k, p in FILES.items():
    print(f"{k:12s}: {p} | exists={p.exists()}")


## 1. Load CAFA Data


In [ ]:
def extract_uniprot_id(header_line: str) -> str:
    token = header_line.strip()
    if token.startswith(">"):
        token = token[1:]
    token = token.split()[0]
    if "|" in token:
        parts = token.split("|")
        if len(parts) >= 2 and parts[1]:
            return parts[1]
    return token

def load_fasta(filepath: Path) -> dict[str, str]:
    sequences = {}
    current_id = None
    current_seq = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = extract_uniprot_id(line)
                current_seq = []
            else:
                current_seq.append(line.upper())
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

sequences = load_fasta(FILES["train_seq"])
train_terms = pd.read_csv(FILES["train_terms"], sep="\t", header=None, names=["EntryID", "GO_Term", "Aspect"])
train_tax = pd.read_csv(FILES["train_tax"], sep="\t", header=None, names=["EntryID", "Taxonomy"])
ia_scores = pd.read_csv(FILES["ia"], sep="\t", header=None, names=["GO_Term", "IA_Score"]) if FILES["ia"].exists() else pd.DataFrame(columns=["GO_Term", "IA_Score"])

annotated_ids = sorted(set(train_terms["EntryID"]).intersection(sequences))
print(f"sequences={len(sequences):,} train_terms={len(train_terms):,} annotated_ids={len(annotated_ids):,}")
display(train_terms.head())


## 2. Parse GO Ontology Graph


In [ ]:
def parse_go_obo(path: Path):
    parents = defaultdict(set)
    children = defaultdict(set)
    namespace = {}
    name = {}
    current_id = None
    current_ns = None
    current_name = None
    obsolete = False

    def flush():
        if current_id and not obsolete:
            if current_ns:
                namespace[current_id] = current_ns
            if current_name:
                name[current_id] = current_name

    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()
            if line == "[Term]":
                flush()
                current_id = None
                current_ns = None
                current_name = None
                obsolete = False
                continue
            if not line or line.startswith("!"):
                continue
            if line.startswith("id: GO:"):
                current_id = line.split("id: ", 1)[1]
            elif line.startswith("name: "):
                current_name = line.split("name: ", 1)[1]
            elif line.startswith("namespace: "):
                current_ns = line.split("namespace: ", 1)[1]
            elif line.startswith("is_obsolete: true"):
                obsolete = True
            elif current_id and line.startswith("is_a: GO:"):
                p = line.split()[1]
                parents[current_id].add(p)
                children[p].add(current_id)
            elif current_id and line.startswith("relationship: part_of GO:"):
                p = line.split()[2]
                parents[current_id].add(p)
                children[p].add(current_id)
        flush()
    return dict(parents), dict(children), namespace, name

go_parents, go_children, go_namespace, go_name = parse_go_obo(FILES["obo"])
namespace_to_aspect = {
    "molecular_function": "F",
    "biological_process": "P",
    "cellular_component": "C",
}
term_to_aspect = train_terms.groupby("GO_Term")["Aspect"].agg(lambda s: s.mode().iloc[0]).to_dict()
for term, ns in go_namespace.items():
    term_to_aspect.setdefault(term, namespace_to_aspect.get(ns))

ia_dict = dict(zip(ia_scores["GO_Term"], ia_scores["IA_Score"])) if len(ia_scores) else {}
def ia(term: str) -> float:
    return float(ia_dict.get(term, 1.0))

@lru_cache(maxsize=None)
def ancestors(term: str) -> frozenset[str]:
    out = set()
    for p in go_parents.get(term, []):
        out.add(p)
        out.update(ancestors(p))
    return frozenset(out)

print(f"GO terms with namespace: {len(go_namespace):,}")
print(f"GO terms with parent edges: {len(go_parents):,}")
print(f"IA rows: {len(ia_dict):,}")


## 3. Select Label Universe And Propagate Labels


In [ ]:
label_counts = train_terms["GO_Term"].value_counts()
eligible_terms = label_counts[label_counts >= CONFIG["min_label_freq"]]
selected_terms = eligible_terms.head(CONFIG["max_labels"]).index.tolist()
selected_term_set = set(selected_terms)
term_to_index = {term: i for i, term in enumerate(selected_terms)}

protein_to_terms_raw = train_terms.groupby("EntryID")["GO_Term"].apply(set).to_dict()

def propagate_terms(terms: set[str]) -> set[str]:
    expanded = set(terms)
    if CONFIG["use_label_propagation"]:
        for t in list(terms):
            expanded.update(ancestors(t))
    if not CONFIG["include_ancestors_outside_selected"]:
        expanded &= selected_term_set
    return expanded

protein_to_terms_selected_raw = {
    pid: set(terms).intersection(selected_term_set)
    for pid, terms in protein_to_terms_raw.items()
}
protein_to_terms_selected_graph_all = {
    pid: propagate_terms(set(terms))
    for pid, terms in protein_to_terms_raw.items()
}
# Keep exactly the same usable protein universe as cafa6.ipynb:
# proteins must have at least one raw selected term before graph propagation.
usable_ids = [
    pid for pid in annotated_ids
    if len(protein_to_terms_selected_raw.get(pid, set())) > 0
]
protein_to_terms_selected_graph = {
    pid: protein_to_terms_selected_graph_all.get(pid, set())
    for pid in usable_ids
}

label_stats = pd.DataFrame({
    "setting": ["raw_selected", "graph_propagated"],
    "positive_labels": [
        sum(len(protein_to_terms_selected_raw.get(pid, set())) for pid in usable_ids),
        sum(len(protein_to_terms_selected_graph.get(pid, set())) for pid in usable_ids),
    ],
    "avg_labels_per_protein": [
        np.mean([len(protein_to_terms_selected_raw.get(pid, set())) for pid in usable_ids]),
        np.mean([len(protein_to_terms_selected_graph.get(pid, set())) for pid in usable_ids]),
    ],
})
print(f"selected_terms={len(selected_terms):,} usable_ids={len(usable_ids):,}")
display(label_stats)
display(pd.Series([term_to_aspect.get(t, "UNK") for t in selected_terms]).value_counts().rename("selected_terms_by_aspect"))


In [ ]:
def make_split_strata(ids):
    rows = []
    for pid in ids:
        terms = protein_to_terms_raw.get(pid, set())
        aspects = sorted({term_to_aspect.get(t, "UNK") for t in terms})
        aspect_sig = "".join(a for a in ["MF", "BP", "CC"] if a in aspects) or "UNK"
        n = len(terms)
        bin_name = "few" if n <= 3 else "mid" if n <= 10 else "many"
        rows.append(f"{aspect_sig}_{bin_name}")
    s = pd.Series(rows, index=ids)
    counts = s.value_counts()
    return s.where(s.map(counts) >= 5, "rare").loc[ids].values

split_output_dir = ARTIFACT_DIR / "splits"
split_output_dir.mkdir(parents=True, exist_ok=True)

strata = make_split_strata(usable_ids)
try:
    train_valid_ids, test_ids = train_test_split(
        usable_ids,
        test_size=CONFIG["test_size"],
        random_state=SEED,
        stratify=strata if CONFIG["split_mode"] == "stratified_protein" else None,
    )
    valid_relative_size = CONFIG["valid_size"] / (1.0 - CONFIG["test_size"])
    train_valid_strata = make_split_strata(train_valid_ids)
    train_ids, valid_ids = train_test_split(
        train_valid_ids,
        test_size=valid_relative_size,
        random_state=SEED,
        stratify=train_valid_strata if CONFIG["split_mode"] == "stratified_protein" else None,
    )
except ValueError as e:
    print(f"[SPLIT] Stratified split failed: {e}. Falling back to random protein split.")
    train_valid_ids, test_ids = train_test_split(usable_ids, test_size=CONFIG["test_size"], random_state=SEED)
    valid_relative_size = CONFIG["valid_size"] / (1.0 - CONFIG["test_size"])
    train_ids, valid_ids = train_test_split(train_valid_ids, test_size=valid_relative_size, random_state=SEED)

train_ids = sorted(train_ids)
valid_ids = sorted(valid_ids)
test_ids = sorted(test_ids)
assert not set(train_ids).intersection(valid_ids)
assert not set(train_ids).intersection(test_ids)
assert not set(valid_ids).intersection(test_ids)

print(f"[SPLIT] train proteins: {len(train_ids):,} ({len(train_ids)/len(usable_ids):.1%})")
print(f"[SPLIT] valid proteins: {len(valid_ids):,} ({len(valid_ids)/len(usable_ids):.1%})")
print(f"[SPLIT] test proteins : {len(test_ids):,} ({len(test_ids)/len(usable_ids):.1%})")
print(f"[SPLIT] total usable  : {len(usable_ids):,}")

split_id_map = {"train": train_ids, "valid": valid_ids, "test": test_ids}
for split_name, ids in split_id_map.items():
    (split_output_dir / f"{split_name}_ids.txt").write_text("\n".join(ids) + "\n")

baseline_split_dir = Path(CONFIG["baseline_split_dir"])
if baseline_split_dir.exists():
    for split_name, ids in split_id_map.items():
        baseline_path = baseline_split_dir / f"{split_name}_ids.txt"
        if baseline_path.exists():
            baseline_ids = baseline_path.read_text().splitlines()
            assert ids == baseline_ids, f"{split_name} split differs from baseline: {len(ids)} vs {len(baseline_ids)}"
    print(f"[SPLIT] Verified exact split match with baseline: {baseline_split_dir}")

def build_label_matrix(ids, protein_to_terms):
    Y = np.zeros((len(ids), len(selected_terms)), dtype=np.float32)
    for i, pid in enumerate(ids):
        for term in protein_to_terms.get(pid, set()):
            j = term_to_index.get(term)
            if j is not None:
                Y[i, j] = 1.0
    return Y

Y_train_raw = build_label_matrix(train_ids, protein_to_terms_selected_raw)
Y_valid_raw = build_label_matrix(valid_ids, protein_to_terms_selected_raw)
Y_test_raw = build_label_matrix(test_ids, protein_to_terms_selected_raw)

Y_train = build_label_matrix(train_ids, protein_to_terms_selected_graph)
Y_valid = build_label_matrix(valid_ids, protein_to_terms_selected_graph)
Y_test = build_label_matrix(test_ids, protein_to_terms_selected_graph)

print(f"Y_train graph: {Y_train.shape}, positives={int(Y_train.sum()):,}, density={Y_train.mean():.5f}")
print(f"Y_valid graph: {Y_valid.shape}, positives={int(Y_valid.sum()):,}, density={Y_valid.mean():.5f}")
print(f"Y_test  graph: {Y_test.shape}, positives={int(Y_test.sum()):,}, density={Y_test.mean():.5f}")

AA = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i + 1 for i, aa in enumerate(AA)}  # 0 = pad/unknown

def encode_sequence(seq, max_len=None):
    if max_len is None:
        max_len = CONFIG["sequence_max_length"]
    arr = np.zeros(max_len, dtype=np.int64)
    for i, aa in enumerate(seq[:max_len]):
        arr[i] = aa_to_idx.get(aa, 0)
    return arr

def build_sequence_matrix(ids):
    X = np.zeros((len(ids), CONFIG["sequence_max_length"]), dtype=np.int64)
    for i, pid in enumerate(ids):
        X[i] = encode_sequence(sequences[pid])
    return X

Xseq_train = build_sequence_matrix(train_ids)
Xseq_valid = build_sequence_matrix(valid_ids)
Xseq_test = build_sequence_matrix(test_ids)
print(f"[SEQ] Xseq_train={Xseq_train.shape}, Xseq_valid={Xseq_valid.shape}, Xseq_test={Xseq_test.shape}")


## 4. Graph Edges, IA Weights, And Training Weights


In [ ]:
# Edges only among selected terms: child -> parent.
selected_edges = []
for child in selected_terms:
    child_idx = term_to_index[child]
    for parent in go_parents.get(child, []):
        parent_idx = term_to_index.get(parent)
        if parent_idx is not None:
            selected_edges.append((child_idx, parent_idx, child, parent))

edge_df = pd.DataFrame(selected_edges, columns=["child_idx", "parent_idx", "child", "parent"])
print(f"selected child->parent edges: {len(edge_df):,}")
display(edge_df.head())

pos_counts = Y_train.sum(axis=0)
pos_weight = (Y_train.shape[0] - pos_counts) / np.maximum(pos_counts, 1.0)
pos_weight = np.sqrt(pos_weight)
pos_weight = np.clip(pos_weight, 1.0, CONFIG["pos_weight_clip"]).astype(np.float32)

ia_values = np.array([ia(t) for t in selected_terms], dtype=np.float32)
ia_values = np.nan_to_num(ia_values, nan=1.0, posinf=1.0, neginf=1.0)
ia_loss_weight = np.power(np.maximum(ia_values, 1e-6), CONFIG["ia_loss_power"])
ia_loss_weight = ia_loss_weight / max(float(np.mean(ia_loss_weight)), 1e-6)
ia_loss_weight = np.clip(ia_loss_weight, 0.25, CONFIG["ia_weight_clip"]).astype(np.float32)

term_weight_df = pd.DataFrame({
    "GO_Term": selected_terms,
    "Name": [go_name.get(t, "") for t in selected_terms],
    "Aspect": [term_to_aspect.get(t, "UNK") for t in selected_terms],
    "Train_Positive_Count": pos_counts.astype(int),
    "Pos_Weight": pos_weight,
    "IA": ia_values,
    "IA_Loss_Weight": ia_loss_weight,
    "Num_Selected_Parents": [len([p for p in go_parents.get(t, []) if p in selected_term_set]) for t in selected_terms],
    "Num_Selected_Children": [len([c for c in go_children.get(t, []) if c in selected_term_set]) for t in selected_terms],
})
display(term_weight_df.sort_values("IA_Loss_Weight", ascending=False).head(20))
term_weight_df.to_csv(ARTIFACT_DIR / "term_training_weights.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(term_weight_df["Pos_Weight"], bins=60, ax=axes[0])
axes[0].set_title("Positive class weights")
sns.histplot(term_weight_df["IA"], bins=60, ax=axes[1])
axes[1].set_title("IA scores")
sns.histplot(term_weight_df["IA_Loss_Weight"], bins=60, ax=axes[2])
axes[2].set_title("IA loss weights")
plt.tight_layout()
plt.show()


## 5. ESM Embeddings


In [ ]:
def prepare_lm_sequences(batch_seqs, model_name):
    if "prot_bert" in model_name.lower() or "protbert" in model_name.lower():
        cleaned = [re.sub(r"[UZOB]", "X", seq.upper()) for seq in batch_seqs]
        return [" ".join(list(seq)) for seq in cleaned]
    return [seq.upper() for seq in batch_seqs]

def pool_lm_outputs(outputs, attention_mask, pooling="mean"):
    hidden = outputs.last_hidden_state
    mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
    if pooling == "cls":
        return hidden[:, 0]
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

def extract_embeddings(ids, split_name):
    cache_path = ARTIFACT_DIR / f"{split_name}_esm_embeddings.npy"
    if CONFIG["cache_embeddings"] and cache_path.exists():
        print(f"loading cached {cache_path}")
        return np.load(cache_path)

    from transformers import AutoModel, AutoTokenizer
    model_name = CONFIG["embedding_model_name"]
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    if DEVICE == "cuda" and CONFIG["use_fp16_embeddings"]:
        model.half()

    vectors = []
    batch_size = CONFIG["embedding_batch_size"]
    max_length = CONFIG["embedding_max_length"]
    with torch.no_grad():
        for start in range(0, len(ids), batch_size):
            batch_ids = ids[start:start + batch_size]
            seqs = [sequences[pid][:max_length] for pid in batch_ids]
            seqs = prepare_lm_sequences(seqs, model_name)
            toks = tokenizer(seqs, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
            toks = {k: v.to(DEVICE) for k, v in toks.items()}
            if DEVICE == "cuda":
                with torch.autocast(device_type="cuda", enabled=bool(CONFIG["use_fp16_embeddings"])):
                    pooled = pool_lm_outputs(model(**toks), toks["attention_mask"], CONFIG["embedding_pooling"])
            else:
                pooled = pool_lm_outputs(model(**toks), toks["attention_mask"], CONFIG["embedding_pooling"])
            vectors.append(pooled.float().cpu().numpy())
            if (start // batch_size) % 25 == 0:
                print(f"{split_name}: {start + len(batch_ids):,}/{len(ids):,}")

    X = np.vstack(vectors).astype(np.float32)
    if CONFIG["cache_embeddings"]:
        np.save(cache_path, X)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return X

X_train = extract_embeddings(train_ids, "train")
X_valid = extract_embeddings(valid_ids, "valid")
X_test = extract_embeddings(test_ids, "test")
print(X_train.shape, X_valid.shape, X_test.shape)


## 6. Graph-Aware Model Architectures


In [ ]:
class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class ProtCNN(nn.Module):
    def __init__(self, output_dim, vocab_size=21, emb_dim=128, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.conv3 = nn.Sequential(nn.Conv1d(emb_dim, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU())
        self.conv5 = nn.Sequential(nn.Conv1d(emb_dim, 256, 5, padding=2), nn.BatchNorm1d(256), nn.ReLU())
        self.conv7 = nn.Sequential(nn.Conv1d(emb_dim, 256, 7, padding=3), nn.BatchNorm1d(256), nn.ReLU())
        self.conv = nn.Sequential(
            nn.Conv1d(768, 512, 3, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, output_dim),
        )

    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = torch.cat([self.conv3(x), self.conv5(x), self.conv7(x)], dim=1)
        x = self.conv(x)
        gap = torch.mean(x, dim=2)
        gmp = torch.max(x, dim=2).values
        return self.head(torch.cat([gap, gmp], dim=1))

class BiLSTMAttention(nn.Module):
    def __init__(self, output_dim, vocab_size=21, emb_dim=128, hidden=256, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm1 = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.MultiheadAttention(hidden * 2, num_heads=8, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden * 2)
        self.lstm2 = nn.LSTM(hidden * 2, hidden // 2, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, output_dim),
        )

    def forward(self, x):
        pad_mask = x.eq(0)
        x = self.embedding(x)
        x, _ = self.lstm1(x)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=pad_mask)
        x = self.norm(x + self.dropout(attn_out))
        x, _ = self.lstm2(x)
        valid = (~pad_mask).unsqueeze(-1).to(x.dtype)
        gap = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        masked = x.masked_fill(pad_mask.unsqueeze(-1), -1e4)
        gmp = masked.max(dim=1).values
        return self.head(torch.cat([gap, gmp], dim=1))

edge_child_idx = torch.tensor(edge_df["child_idx"].values, dtype=torch.long, device=DEVICE) if len(edge_df) else torch.empty(0, dtype=torch.long, device=DEVICE)
edge_parent_idx = torch.tensor(edge_df["parent_idx"].values, dtype=torch.long, device=DEVICE) if len(edge_df) else torch.empty(0, dtype=torch.long, device=DEVICE)
pos_weight_t = torch.from_numpy(pos_weight).to(DEVICE)
ia_weight_t = torch.from_numpy(ia_loss_weight).to(DEVICE)

def graph_aware_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight_t, reduction="none")
    bce = (bce * ia_weight_t).mean()
    if len(edge_child_idx) == 0 or CONFIG["hierarchy_penalty_weight"] <= 0:
        return bce, {"bce": float(bce.detach().cpu()), "hierarchy": 0.0}

    probs = torch.sigmoid(logits)
    child_idx = edge_child_idx
    parent_idx = edge_parent_idx
    if len(child_idx) > CONFIG["hierarchy_edge_sample"]:
        sample = torch.randint(0, len(child_idx), (CONFIG["hierarchy_edge_sample"],), device=DEVICE)
        child_idx = child_idx[sample]
        parent_idx = parent_idx[sample]
    violation = F.relu(probs[:, child_idx] - probs[:, parent_idx] + CONFIG["hierarchy_margin"])
    hierarchy = violation.mean()
    total = bce + CONFIG["hierarchy_penalty_weight"] * hierarchy
    return total, {"bce": float(bce.detach().cpu()), "hierarchy": float(hierarchy.detach().cpu())}

def make_loader(X, Y, batch_size, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(Y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=(DEVICE == "cuda"))


In [ ]:
def basic_metrics(y_true, y_prob, threshold=0.2):
    y_pred = (y_prob >= threshold).astype(np.uint8)
    return {
        "threshold": threshold,
        "micro_f1": f1_score(y_true.ravel(), y_pred.ravel(), zero_division=0),
        "micro_precision": precision_score(y_true.ravel(), y_pred.ravel(), zero_division=0),
        "micro_recall": recall_score(y_true.ravel(), y_pred.ravel(), zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "avg_pred_terms": float(y_pred.sum(axis=1).mean()),
    }

def weighted_micro_metrics(y_true, y_prob, term_weights, threshold=0.2):
    y_pred = (y_prob >= threshold).astype(np.uint8)
    w = term_weights.reshape(1, -1)
    tp = float((((y_true == 1) & (y_pred == 1)).astype(np.float32) * w).sum())
    fp = float((((y_true == 0) & (y_pred == 1)).astype(np.float32) * w).sum())
    fn = float((((y_true == 1) & (y_pred == 0)).astype(np.float32) * w).sum())
    pr = tp / max(tp + fp, 1e-12)
    rc = tp / max(tp + fn, 1e-12)
    f1 = 2 * pr * rc / max(pr + rc, 1e-12)
    return {"weighted_micro_precision": pr, "weighted_micro_recall": rc, "weighted_micro_f1": f1}

def hierarchy_violation_rate(y_prob):
    if len(edge_df) == 0:
        return {"hierarchy_violation_rate": 0.0, "hierarchy_violation_mean": 0.0}
    child = edge_df["child_idx"].values
    parent = edge_df["parent_idx"].values
    diff = y_prob[:, child] - y_prob[:, parent]
    viol = diff > 1e-8
    return {
        "hierarchy_violation_rate": float(viol.mean()),
        "hierarchy_violation_mean": float(np.maximum(diff, 0).mean()),
    }

def aspect_metrics(y_true, y_prob, threshold, label=""):
    rows = []
    aspects = np.array([term_to_aspect.get(t, "UNK") for t in selected_terms])
    for aspect in sorted(set(aspects)):
        idx = np.where(aspects == aspect)[0]
        if len(idx) == 0:
            continue
        m = basic_metrics(y_true[:, idx], y_prob[:, idx], threshold)
        m.update({"aspect": aspect, "label": label, "num_terms": len(idx)})
        rows.append(m)
    return pd.DataFrame(rows)

@torch.no_grad()
def predict_proba(model, X, batch_size=512):
    model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=batch_size, shuffle=False)
    chunks = []
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        chunks.append(torch.sigmoid(model(xb)).cpu().numpy().astype(np.float32))
    return np.vstack(chunks)

def tune_thresholds(y_true, y_prob, label):
    rows = []
    for th in CONFIG["threshold_grid"]:
        m = basic_metrics(y_true, y_prob, th)
        m.update(weighted_micro_metrics(y_true, y_prob, ia_values, th))
        m.update(hierarchy_violation_rate(y_prob))
        m["label"] = label
        rows.append(m)
    return pd.DataFrame(rows)


In [ ]:
BRANCH_CKPT_DIR = ARTIFACT_DIR / "branch_checkpoints"
BRANCH_CKPT_DIR.mkdir(parents=True, exist_ok=True)

def train_graph_model(model, train_loader, valid_X, model_name, epochs, predict_batch_size=512):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    best_state = None
    best_f1 = -1.0
    bad_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        losses, bces, hrels = [], [], []
        start = time.time()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss, parts = graph_aware_loss(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
            bces.append(parts["bce"])
            hrels.append(parts["hierarchy"])

        y_prob_valid_branch = predict_proba(model, valid_X, batch_size=predict_batch_size)
        valid_metric = basic_metrics(Y_valid, y_prob_valid_branch, threshold=0.10)
        valid_metric.update(weighted_micro_metrics(Y_valid, y_prob_valid_branch, ia_values, threshold=0.10))
        valid_metric.update(hierarchy_violation_rate(y_prob_valid_branch))
        valid_metric.update({
            "model": model_name,
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_bce": float(np.mean(bces)),
            "train_hierarchy_penalty": float(np.mean(hrels)),
            "epoch_seconds": time.time() - start,
            "lr": optimizer.param_groups[0]["lr"],
        })
        history.append(valid_metric)
        scheduler.step(valid_metric["micro_f1"])

        print(
            f"[{model_name}] epoch={epoch:02d} loss={valid_metric['train_loss']:.4f} "
            f"bce={valid_metric['train_bce']:.4f} hier={valid_metric['train_hierarchy_penalty']:.5f} "
            f"valid_f1@0.10={valid_metric['micro_f1']:.4f} "
            f"valid_w_f1@0.10={valid_metric['weighted_micro_f1']:.4f} "
            f"viol={valid_metric['hierarchy_violation_rate']:.4f}"
        )
        if valid_metric["micro_f1"] > best_f1:
            best_f1 = valid_metric["micro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= CONFIG["early_stopping_patience"]:
                print(f"[{model_name}] early stopping")
                break

    if best_state:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), {k: v.cpu() for k, v in model.state_dict().items()}

def save_branch_checkpoint(branch_name, state_dict):
    path = BRANCH_CKPT_DIR / f"{branch_name}.pt"
    torch.save({
        "branch": branch_name,
        "state_dict": state_dict,
        "config": CONFIG,
        "selected_terms": selected_terms,
        "graph_aware": True,
    }, path)
    print(f"[CHECKPOINT] saved {branch_name}: {path} ({path.stat().st_size / 1024 / 1024:.1f} MB)")

predictions_valid_raw = {}
predictions_test_raw = {}
histories = []
model_states = {}

if CONFIG["train_esm_mlp"]:
    print("\n[PIPELINE] Training graph-aware ESM-MLP")
    train_loader = make_loader(X_train, Y_train, CONFIG["esm_batch_size"], True)
    esm_mlp = EmbeddingMLP(X_train.shape[1], len(selected_terms), CONFIG["esm_hidden_dims"], CONFIG["esm_dropout"])
    esm_mlp, hist, state = train_graph_model(esm_mlp, train_loader, X_valid, "esm_mlp", CONFIG["esm_epochs"], CONFIG["esm_batch_size"])
    histories.append(hist)
    predictions_valid_raw["esm_mlp"] = predict_proba(esm_mlp, X_valid, CONFIG["esm_batch_size"])
    predictions_test_raw["esm_mlp"] = predict_proba(esm_mlp, X_test, CONFIG["esm_batch_size"])
    model_states["esm_mlp"] = state
    save_branch_checkpoint("esm_mlp", state)
    del esm_mlp
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if CONFIG["train_protcnn"]:
    print("\n[PIPELINE] Training graph-aware ProtCNN")
    train_loader = make_loader(Xseq_train, Y_train, CONFIG["sequence_batch_size"], True)
    protcnn = ProtCNN(len(selected_terms), dropout=CONFIG["protcnn_dropout"])
    protcnn, hist, state = train_graph_model(protcnn, train_loader, Xseq_valid, "protcnn", CONFIG["protcnn_epochs"], CONFIG["sequence_batch_size"])
    histories.append(hist)
    predictions_valid_raw["protcnn"] = predict_proba(protcnn, Xseq_valid, CONFIG["sequence_batch_size"])
    predictions_test_raw["protcnn"] = predict_proba(protcnn, Xseq_test, CONFIG["sequence_batch_size"])
    model_states["protcnn"] = state
    save_branch_checkpoint("protcnn", state)
    del protcnn
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if CONFIG["train_bilstm_attention"]:
    print("\n[PIPELINE] Training graph-aware BiLSTM-Attention")
    seq_bs = max(32, CONFIG["sequence_batch_size"] // 4)
    train_loader = make_loader(Xseq_train, Y_train, seq_bs, True)
    bilstm = BiLSTMAttention(len(selected_terms), dropout=CONFIG["bilstm_dropout"])
    bilstm, hist, state = train_graph_model(bilstm, train_loader, Xseq_valid, "bilstm_attention", CONFIG["bilstm_epochs"], seq_bs)
    histories.append(hist)
    predictions_valid_raw["bilstm_attention"] = predict_proba(bilstm, Xseq_valid, seq_bs)
    predictions_test_raw["bilstm_attention"] = predict_proba(bilstm, Xseq_test, seq_bs)
    model_states["bilstm_attention"] = state
    save_branch_checkpoint("bilstm_attention", state)
    del bilstm
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if not predictions_valid_raw:
    raise RuntimeError("No branches were trained. Enable at least one train_* flag in CONFIG.")

active_weights = {k: CONFIG["ensemble_weights"].get(k, 1.0) for k in predictions_valid_raw}
weight_sum = sum(active_weights.values())
active_weights = {k: v / weight_sum for k, v in active_weights.items()}
y_prob_valid_raw = sum(active_weights[k] * predictions_valid_raw[k] for k in predictions_valid_raw).astype(np.float32)
y_prob_test_raw = sum(active_weights[k] * predictions_test_raw[k] for k in predictions_test_raw).astype(np.float32)
print(f"[ENSEMBLE] active weights: {active_weights}")

history_df = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
history_df.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)
display(history_df.tail())


## 7. Graph-Aware Prediction Propagation


In [ ]:
selected_parent_indices = [
    [term_to_index[p] for p in go_parents.get(t, []) if p in term_to_index]
    for t in selected_terms
]

def propagate_scores_to_parents(probs, passes=8):
    out = probs.copy()
    for _ in range(passes):
        changed = False
        for child_idx, parent_idxs in enumerate(selected_parent_indices):
            if not parent_idxs:
                continue
            child_scores = out[:, child_idx]
            before = out[:, parent_idxs].copy()
            out[:, parent_idxs] = np.maximum(out[:, parent_idxs], child_scores[:, None])
            if not np.array_equal(before, out[:, parent_idxs]):
                changed = True
        if not changed:
            break
    return out

y_prob_valid_graph = propagate_scores_to_parents(y_prob_valid_raw)
y_prob_test_graph = propagate_scores_to_parents(y_prob_test_raw)

print("raw valid violations:", hierarchy_violation_rate(y_prob_valid_raw))
print("propagated valid violations:", hierarchy_violation_rate(y_prob_valid_graph))


## 8. Evaluation: Raw vs Graph Labels, Raw vs Propagated Scores


In [ ]:
eval_tables = []
for label, y_true, y_prob in [
    ("valid_raw_labels_raw_scores", Y_valid_raw, y_prob_valid_raw),
    ("valid_graph_labels_raw_scores", Y_valid, y_prob_valid_raw),
    ("valid_graph_labels_propagated_scores", Y_valid, y_prob_valid_graph),
    ("test_raw_labels_raw_scores", Y_test_raw, y_prob_test_raw),
    ("test_graph_labels_raw_scores", Y_test, y_prob_test_raw),
    ("test_graph_labels_propagated_scores", Y_test, y_prob_test_graph),
]:
    eval_tables.append(tune_thresholds(y_true, y_prob, label))

threshold_df = pd.concat(eval_tables, ignore_index=True)
threshold_df.to_csv(ARTIFACT_DIR / "threshold_tuning_graph_aware.csv", index=False)
display(threshold_df.sort_values(["label", "threshold"]).head())

valid_select = threshold_df[threshold_df["label"] == "valid_graph_labels_propagated_scores"]
best_row = valid_select.loc[valid_select["micro_f1"].idxmax()].to_dict()
best_threshold = float(best_row["threshold"])
print("best threshold selected on valid graph propagated:", best_threshold)
display(pd.DataFrame([best_row]))

final_rows = []
for label, y_true, y_prob in [
    ("valid_raw_labels_raw_scores", Y_valid_raw, y_prob_valid_raw),
    ("valid_graph_labels_raw_scores", Y_valid, y_prob_valid_raw),
    ("valid_graph_labels_propagated_scores", Y_valid, y_prob_valid_graph),
    ("test_raw_labels_raw_scores", Y_test_raw, y_prob_test_raw),
    ("test_graph_labels_raw_scores", Y_test, y_prob_test_raw),
    ("test_graph_labels_propagated_scores", Y_test, y_prob_test_graph),
]:
    m = basic_metrics(y_true, y_prob, best_threshold)
    m.update(weighted_micro_metrics(y_true, y_prob, ia_values, best_threshold))
    m.update(hierarchy_violation_rate(y_prob))
    m["label"] = label
    final_rows.append(m)

final_metrics_df = pd.DataFrame(final_rows)
final_metrics_df.to_csv(ARTIFACT_DIR / "final_metrics_graph_aware.csv", index=False)
display(final_metrics_df)


In [ ]:
aspect_df = pd.concat([
    aspect_metrics(Y_valid, y_prob_valid_graph, best_threshold, "valid_graph_labels_propagated_scores"),
    aspect_metrics(Y_test, y_prob_test_graph, best_threshold, "test_graph_labels_propagated_scores"),
], ignore_index=True)
aspect_df.to_csv(ARTIFACT_DIR / "aspect_metrics_graph_aware.csv", index=False)
display(aspect_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.lineplot(data=threshold_df[threshold_df["label"].str.startswith("valid")], x="threshold", y="micro_f1", hue="label", marker="o", ax=axes[0])
axes[0].set_title("Validation micro F1")
sns.lineplot(data=threshold_df[threshold_df["label"].str.startswith("valid")], x="threshold", y="weighted_micro_f1", hue="label", marker="o", ax=axes[1])
axes[1].set_title("Validation IA-weighted micro F1")
sns.barplot(data=final_metrics_df, x="label", y="hierarchy_violation_rate", ax=axes[2])
axes[2].set_title("Hierarchy violation rate")
axes[2].tick_params(axis="x", rotation=80)
plt.tight_layout()
plt.show()


## 9. Inspect Top Predictions


In [ ]:
def top_predictions_for_row(row_probs, top_k=20):
    idx = np.argsort(-row_probs)[:top_k]
    return pd.DataFrame([
        {
            "GO_Term": selected_terms[j],
            "Aspect": term_to_aspect.get(selected_terms[j]),
            "Name": go_name.get(selected_terms[j], ""),
            "Score": float(row_probs[j]),
            "Train_Frequency": int(label_counts.get(selected_terms[j], 0)),
            "IA": ia(selected_terms[j]),
            "Num_Selected_Parents": len(selected_parent_indices[j]),
        }
        for j in idx
    ])

for sample_i in range(min(3, len(test_ids))):
    pid = test_ids[sample_i]
    print("=" * 100)
    print(f"Test protein: {pid} | length={len(sequences[pid])}")
    display(top_predictions_for_row(y_prob_test_graph[sample_i], 20))


## 10. Save Artifacts And Official Evaluator Inputs


In [ ]:
torch.save({"model_states": model_states, "config": CONFIG, "active_ensemble_weights": active_weights}, ARTIFACT_DIR / "graph_aware_models.pt")
with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump({**CONFIG, "best_threshold_micro_f1": best_threshold, "active_ensemble_weights": active_weights}, f, indent=2)
with open(ARTIFACT_DIR / "go_terms.json", "w") as f:
    json.dump(selected_terms, f, indent=2)
with open(ARTIFACT_DIR / "go_metadata.json", "w") as f:
    json.dump({
        "term_to_aspect": {t: term_to_aspect.get(t) for t in selected_terms},
        "term_name": {t: go_name.get(t, "") for t in selected_terms},
        "ia": {t: ia(t) for t in selected_terms},
        "parents": {t: sorted([p for p in go_parents.get(t, []) if p in selected_term_set]) for t in selected_terms},
    }, f, indent=2)

np.save(ARTIFACT_DIR / "valid_probabilities_raw.npy", y_prob_valid_raw.astype(np.float32))
np.save(ARTIFACT_DIR / "valid_probabilities_graph_propagated.npy", y_prob_valid_graph.astype(np.float32))
np.save(ARTIFACT_DIR / "test_probabilities_raw.npy", y_prob_test_raw.astype(np.float32))
np.save(ARTIFACT_DIR / "test_probabilities_graph_propagated.npy", y_prob_test_graph.astype(np.float32))

official_eval_dir = ARTIFACT_DIR / "official_eval"
pred_dir = official_eval_dir / "predictions"
valid_pred_dir = official_eval_dir / "predictions_valid"
pred_dir.mkdir(parents=True, exist_ok=True)
valid_pred_dir.mkdir(parents=True, exist_ok=True)

(official_eval_dir / "train_ids.txt").write_text("\n".join(train_ids) + "\n")
(official_eval_dir / "valid_ids.txt").write_text("\n".join(valid_ids) + "\n")
(official_eval_dir / "test_ids.txt").write_text("\n".join(test_ids) + "\n")
(official_eval_dir / "terms_of_interest.tsv").write_text("\n".join(selected_terms) + "\n")

def write_ground_truth(ids, protein_to_terms, output_path):
    rows = []
    for pid in ids:
        for term in sorted(protein_to_terms.get(pid, set()).intersection(selected_term_set)):
            rows.append((pid, term))
    pd.DataFrame(rows).to_csv(output_path, sep="\t", header=False, index=False)
    return len(rows)

def write_prediction_tsv(ids, probs, output_path):
    rows = []
    for i, pid in enumerate(ids):
        order = np.argsort(-probs[i])[:CONFIG["official_eval_top_k"]]
        for j in order:
            score = float(probs[i, j])
            if score > 0:
                rows.append((pid, selected_terms[j], min(score, 1.0)))
    pd.DataFrame(rows).to_csv(output_path, sep="\t", header=False, index=False)
    return len(rows)

write_ground_truth(valid_ids, protein_to_terms_selected_graph, official_eval_dir / "valid_ground_truth_graph.tsv")
write_ground_truth(test_ids, protein_to_terms_selected_graph, official_eval_dir / "ground_truth_graph.tsv")
write_prediction_tsv(valid_ids, y_prob_valid_graph, valid_pred_dir / "graph_aware_valid.tsv")
write_prediction_tsv(test_ids, y_prob_test_graph, pred_dir / "graph_aware.tsv")

outputs = sorted(str(p) for p in ARTIFACT_DIR.rglob("*") if p.is_file())
print("Saved files:")
for p in outputs[:80]:
    print(" -", p)
if len(outputs) > 80:
    print(f"... and {len(outputs) - 80} more")


## Notes

Cải tiến này đưa ontology vào 3 điểm chính:

1. **Targets**: nếu term con là positive thì ancestor trong selected label universe cũng được positive.
2. **Loss**: BCE được nhân theo IA loss weight, cộng penalty khi child score vượt parent score.
3. **Inference**: score được propagate lên parent bằng max, giống luật CAFA evaluator sẽ propagate parent score từ child nếu submission chưa làm.

Để so sánh công bằng với notebook baseline, xem các file:

- `branch_checkpoints/`
- `term_training_weights.csv`
- `threshold_tuning_graph_aware.csv`
- `final_metrics_graph_aware.csv`
- `aspect_metrics_graph_aware.csv`
- `official_eval/`
